

## Ожидаемые результаты
| Категория | v1 результат | Ожидаемо v3 |
|---|---|---|
| XR_FINGER | 0.537 | 0.68–0.73 |
| XR_HAND | 0.565 | 0.70–0.75 |
| XR_SHOULDER | 0.566 | 0.70–0.74 |
| XR_HUMERUS | 0.716 | 0.78–0.83 |
| OVERALL | 0.635 | 0.76–0.82 |

## Шаг 1 — GPU и библиотеки

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не найден!'


GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
DEVICE   = 'cuda'

print(f'GPU:  {GPU_NAME}')
print(f'VRAM: {VRAM_GB:.1f} GB')

# DINOv2-Large занимает ~6GB весов
# Адаптеры + головы ещё ~0.5GB
# Активации при batch=4 для 448x448 требуют меньше VRAM
BATCH_SIZE = 64
print(f'Batch size: {BATCH_SIZE}')


GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Batch size: 64


In [ ]:
%%capture
!pip install timm transformers scikit-learn pandas matplotlib seaborn albumentations opencv-python-headless

In [ ]:
import os, json, shutil, time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

from transformers import AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    cohen_kappa_score, roc_auc_score, f1_score, accuracy_score
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.backends.cudnn.benchmark = True
print('Готово')

Готово


## Шаг 2 — Конфигурация

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Пути ─────────────────────────────────────────────────
ARCHIVE_PATH = '/content/drive/MyDrive/MURA-v1.1-resized-518x518.zip'
CKPT_DIR     = '/content/drive/MyDrive/MURA_v3/checkpoints'
RESULTS_DIR  = '/content/drive/MyDrive/MURA_v3/results'
for d in [CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Гиперпараметры ────────────────────────────────────────
IMG_SIZE     = 518
EPOCHS = 18
NUM_WORKERS  = 2
ACCUM_STEPS  = 1


# ImageNet нормализация
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

BONE_CATEGORIES = [
    'XR_WRIST', 'XR_ELBOW', 'XR_SHOULDER',
    'XR_FINGER', 'XR_FOREARM', 'XR_HUMERUS', 'XR_HAND'
]

# Веса loss — смягчены для ProgressiveLoss
BONE_LOSS_WEIGHTS = {
    'XR_FINGER':   1.5,
    'XR_HAND':     2,
    'XR_SHOULDER': 2,
    'XR_FOREARM':  1.2,
    'XR_ELBOW':    1.0,
    'XR_HUMERUS':  1.0,
    'XR_WRIST':    1.0,
}
MEDIUM_BONES = {'XR_SHOULDER'}  # только лёгкие деформации
WEAK_BONES   = {'XR_FINGER', 'XR_HAND', 'XR_FOREARM'}

print('Конфигурация v3 (Simple DINOv2):')
print(f'  DINOv2 Model:  DINOv2-Large (307M, заморожен)')
print(f'  Batch:         {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}')
print(f'  Epochs:        {EPOCHS}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Конфигурация v3 (Simple DINOv2):
  DINOv2 Model:  DINOv2-Large (307M, заморожен)
  Batch:         64 x 1 = 64
  Epochs:        18


## Шаг 3 — Данные

In [ ]:
import zipfile
import os
import shutil
import time
import pandas as pd
from sklearn.model_selection import train_test_split

LOCAL_DATA = '/content/mura_data'

# Проверяем наличие обеих папок (train и valid), иначе перераспаковываем
if os.path.exists(LOCAL_DATA) and 'train' in os.listdir(LOCAL_DATA) and 'valid' in os.listdir(LOCAL_DATA):
    print('Данные уже на локальном диске')
else:
    print('Распаковка...')
    start = time.time()
    if os.path.exists('/content/mura_extracted'):
        shutil.rmtree('/content/mura_extracted')
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall('/content/mura_extracted')

    folders = [f for f in os.listdir('/content/mura_extracted')
               if os.path.isdir(f'/content/mura_extracted/{f}')]

    # Ищем, где реально лежат train и valid
    if 'train' in folders and 'valid' in folders:
        src = '/content/mura_extracted'
    else:
        # Скорее всего внутри вложенной папки (например MURA-v1.1)
        src = f'/content/mura_extracted/{folders[0]}' if folders else '/content/mura_extracted'

    if os.path.exists(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)
    shutil.move(src, LOCAL_DATA)

    # Убираем остатки mura_extracted если move забрал внутреннюю папку
    if os.path.exists('/content/mura_extracted'):
        shutil.rmtree('/content/mura_extracted')

    print(f'Готово за {time.time()-start:.0f} сек')

TRAIN_CSV = f'{LOCAL_DATA}/train.csv'
VAL_CSV   = f'{LOCAL_DATA}/valid.csv'

def build_csv(root_dir, split='train'):
    records, split_dir = [], os.path.join(root_dir, split)
    for bone in os.listdir(split_dir):
        bone_dir = os.path.join(split_dir, bone)
        if not os.path.isdir(bone_dir): continue
        for patient in os.listdir(bone_dir):
            for study in os.listdir(os.path.join(bone_dir, patient)):
                study_dir = os.path.join(bone_dir, patient, study)
                if not os.path.isdir(study_dir): continue
                label = 1 if 'positive' in study.lower() else 0
                for f in os.listdir(study_dir):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        records.append({'image_path': os.path.join(study_dir,f),
                                        'label': label, 'bone_category': bone})
    df = pd.DataFrame(records)
    out = os.path.join(root_dir, f'{split}.csv')
    df.to_csv(out, index=False)
    print(f'  {split}: {len(df):,} снимков')
    return out

# Если файлы еще не созданы, парсим папки
if not os.path.exists(TRAIN_CSV):
    build_csv(LOCAL_DATA, 'train')
    build_csv(LOCAL_DATA, 'valid')

# 1. Загружаем то, что выдал парсер
train_df_full = pd.read_csv(f'{LOCAL_DATA}/train.csv')
test_df       = pd.read_csv(f'{LOCAL_DATA}/valid.csv') # Официальный valid становится Test-ом

# 2. Извлекаем ID пациента для правильного сплита
train_df_full['patient_id'] = train_df_full['image_path'].apply(lambda x: x.split('/')[-3])

# 3. Разбиваем уникальных пациентов (90% обучаются / 10% валидируются)
unique_patients = train_df_full['patient_id'].unique()
train_patients, val_patients = train_test_split(unique_patients, test_size=0.1, random_state=42)

# 4. Формируем 3 чистых датафрейма на основе распределения пациентов
train_df = train_df_full[train_df_full['patient_id'].isin(train_patients)].copy()
val_df   = train_df_full[train_df_full['patient_id'].isin(val_patients)].copy()

# 5. Сохраняем новые правильные CSV
NEW_TRAIN_CSV = f'{LOCAL_DATA}/new_train.csv'
NEW_VAL_CSV   = f'{LOCAL_DATA}/new_val.csv'
NEW_TEST_CSV  = f'{LOCAL_DATA}/new_test.csv'

train_df.to_csv(NEW_TRAIN_CSV, index=False)
val_df.to_csv(NEW_VAL_CSV, index=False)
test_df.to_csv(NEW_TEST_CSV, index=False)

print(f'Train: {len(train_df):,} | Val (Отщепленный): {len(val_df):,} | Test (Stanford): {len(test_df):,}')

Распаковка...
Готово за 51 сек
  train: 36,808 снимков
  valid: 3,197 снимков
Train: 33,181 | Val (Отщепленный): 3,627 | Test (Stanford): 3,197


## Шаг 4 — Dataset с Albumentations (Медицинские аугментации)

In [ ]:
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── Аугментации (Медицинские) ───────────────────────────────────
# Выносим правильный ресайз с паддингом в отдельную переменную для чистоты кода
resize_pad = [
    A.Resize(IMG_SIZE, IMG_SIZE)
]

def get_strong_transform(img_size):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.6),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=0, fill=0, p=0.4),
        A.GridDistortion(num_steps=5, distort_limit=0.08, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.2),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.GaussNoise(p=0.2),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.2),
        *resize_pad,
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_medium_transform(img_size):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.6),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=0, fill=0, p=0.4),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.GaussNoise(p=0.2),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.2),
        *resize_pad,
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_weak_transform(img_size):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.5),
        A.CLAHE(clip_limit=1.5, tile_grid_size=(8, 8), p=0.4),
        A.RandomGamma(gamma_limit=(85, 115), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        *resize_pad,
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transform(img_size):
    return A.Compose([
        *resize_pad,
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size):
    base = [
        *resize_pad,
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ]
    return [
        A.Compose(base),
        A.Compose([A.HorizontalFlip(p=1.0)] + base),
        A.Compose([A.Rotate(limit=(10,10), p=1.0, border_mode=cv2.BORDER_CONSTANT, fill=0)] + base),
        A.Compose([A.Rotate(limit=(-10,-10), p=1.0, border_mode=cv2.BORDER_CONSTANT, fill=0)] + base),
        A.Compose([A.CLAHE(clip_limit=2.0, p=1.0)] + base),
        A.Compose([A.RandomGamma(gamma_limit=(115,115), p=1.0)] + base),
        A.Compose([A.RandomGamma(gamma_limit=(85,85), p=1.0)] + base),
        A.Compose([A.HorizontalFlip(p=1.0), A.CLAHE(clip_limit=2.0, p=1.0)] + base),
    ]

tta_transforms = get_tta_transforms(IMG_SIZE)

class MURADataset(Dataset):
    def __init__(self, csv_path, is_train=False, tta_idx=None):
        self.df       = pd.read_csv(csv_path)
        self.is_train = is_train
        self.tta_idx  = tta_idx
        self.strong_t = get_strong_transform(IMG_SIZE)
        self.medium_t = get_medium_transform(IMG_SIZE)
        self.weak_t   = get_weak_transform(IMG_SIZE)
        self.val_t    = get_val_transform(IMG_SIZE)
        self.tta_t    = tta_transforms

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img_path = row['image_path']
        img  = cv2.imread(img_path)

        # Защита от битых файлов: если картинка не прочиталась, возвращаем черную заглушку
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        bone = str(row['bone_category'])

        # Извлекаем Study ID (например: patient11185/study1_positive)
        # Путь MURA: MURA-v1.1/train/XR_ELBOW/patient11185/study1_positive/image1.png
        parts = img_path.replace('\\', '/').split('/')
        study_id = f"{parts[-3]}/{parts[-2]}"

        if self.tta_idx is not None:
            t = self.tta_t[self.tta_idx]
        elif self.is_train:
            if bone in WEAK_BONES:
                t = self.strong_t
            elif bone in MEDIUM_BONES:
                t = self.medium_t
            else:
                t = self.weak_t
        else:
            t = self.val_t

        augmented = t(image=img)

        # Возвращаем 4 элемента!
        return augmented['image'], int(row['label']), bone, study_id

# WeightedSampler — слабые кости встречаются чаще
sample_weights = torch.DoubleTensor([
    BONE_LOSS_WEIGHTS.get(r['bone_category'], 1.0)
    for _, r in train_df.iterrows()
])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Загружаем Train и Val в DataLoaders для процесса обучения
train_dataset = MURADataset(NEW_TRAIN_CSV, is_train=True)
val_dataset   = MURADataset(NEW_VAL_CSV,   is_train=False) # На нем работает Early Stopping

# Спрятанный "сундук" (использовать ТОЛЬКО после обучения для финальной проверки)
test_dataset  = MURADataset(NEW_TEST_CSV,  is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=NUM_WORKERS, pin_memory=True)

pos = train_df['label'].sum()
neg = len(train_df) - pos
POS_WEIGHT = torch.tensor([neg/pos]).to(DEVICE)

print(f'Train: {len(train_loader)} батчей | Val: {len(val_loader)} батчей | Test: {len(test_loader)} батчей')
print(f'POS_WEIGHT: {POS_WEIGHT.item():.3f}')


Train: 2074 батчей | Val: 227 батчей | Test: 200 батчей
POS_WEIGHT: 1.473


## Шаг 5 — Архитектура: DINOv2

In [ ]:
from transformers import AutoModel
import torch.nn as nn
import torch

class SimpleMURA(nn.Module):
    def __init__(self, bone_categories):
        super().__init__()

        print('Загрузка DINOv2-Large...')
        # DINOv2-Large — изначально заморожен полностью
        self.dino = AutoModel.from_pretrained('facebook/dinov2-large')
        for p in self.dino.parameters():
            p.requires_grad = False

        # ИЗМЕНЕНИЕ 1: Вход теперь 2048 (1024 от CLS + 1024 от усредненных патчей)
        self.head = nn.Sequential(
            nn.Linear(2048, 512),       # Расширили слой под новые признаки
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 64),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.classifiers = nn.ModuleDict({
            bone: nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(64, 2)
            )
            for bone in bone_categories
        })
        print('Модель инициализирована (DINOv2-Large заморожен). Включен Patch Pooling.')

    def forward(self, x, bone):
        # Получаем все выходы от DINOv2
        outputs = self.dino(x)

        # 1. Глобальный признак (CLS-токен) - семантика кости
        cls_token = outputs.last_hidden_state[:, 0, :]

        # 2. Локальные признаки (Токены патчей) - текстура и микротрещины
        patch_tokens = outputs.last_hidden_state[:, 1:, :]

        # Усредняем токены патчей (Global Average Pooling)
        patch_pooled = patch_tokens.mean(dim=1)

        # 3. Соединяем глобальное и локальное зрение
        combined_features = torch.cat([cls_token, patch_pooled], dim=1) # Размер станет 2048

        feat = self.head(combined_features)
        return self.classifiers[bone](feat)

    def unfreeze_last_n(self, n, optimizer):
        """Разморозить последние N блоков И добавить в optimizer."""
        layers = list(self.dino.encoder.layer)

        # Разморозить
        for layer in layers[-n:]:
            for p in layer.parameters():
                p.requires_grad = True

        # Найти новые параметры
        existing = {id(p) for g in optimizer.param_groups for p in g['params']}
        new_params = [p for p in self.parameters()
                      if p.requires_grad and id(p) not in existing]

        # Добавить в optimizer
        if new_params:
            optimizer.add_param_group({
                'params': new_params,
                'lr':     8e-6,
                'name':  f'dino_last_{n}'
            })

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad) / 1e6
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"  Разморожено {n} блоков")
        print(f"  Обучаемых: {trainable:.1f}M")
        print(f"  VRAM: {vram:.1f}GB")
        return new_params

model = SimpleMURA(BONE_CATEGORIES).to(DEVICE)
vram = torch.cuda.memory_allocated() / 1e9
print(f'VRAM после загрузки: {vram:.1f}GB')


Загрузка DINOv2-Large...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

Модель инициализирована (DINOv2-Large заморожен). Включен Patch Pooling.
VRAM после загрузки: 1.2GB


## Шаг 6 — Оптимизатор и Scheduler

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
import shutil

class ProgressiveLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma      = gamma
        self.smoothing  = smoothing
        self.epoch      = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def bce_loss(self, logits, labels):
        # Использование CrossEntropy для совместимости с focal_loss (избегаем градиентного шока)
        weights = torch.tensor([1.0, self.pos_weight], dtype=torch.float, device=logits.device)
        return F.cross_entropy(
            logits.float(),
            labels,
            weight=weights
            # УБРАЛИ label_smoothing, чтобы соответствовать Focal Loss
        )

    def focal_loss(self, logits, labels):
        ce      = F.cross_entropy(logits.float(), labels, reduction='none')
        pt      = torch.exp(-ce)
        alpha_t = torch.where(labels == 1, torch.full_like(ce, self.pos_weight), torch.ones_like(ce))
        return (alpha_t * (1 - pt) ** self.gamma * ce).mean()

    def forward(self, logits, labels):
        if self.epoch < 5:
            return self.bce_loss(logits, labels)
        elif self.epoch < 9:
            alpha = (self.epoch - 5) / 4.0
            return (1 - alpha) * self.bce_loss(logits, labels) + alpha * self.focal_loss(logits, labels)
        else:
            return self.focal_loss(logits, labels)

criterion = ProgressiveLoss(pos_weight=POS_WEIGHT.item(), gamma=1.5, smoothing=0.1)

# Только head и classifiers обучаются изначально
optimizer = torch.optim.AdamW([
    {'params': model.head.parameters(), 'lr': 3e-4, 'name': 'head'},        # голова учится быстро
    {'params': model.classifiers.parameters(), 'lr': 3e-4, 'name': 'cls'},
], weight_decay=0.01)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_kappa = 0.0
        self.counter    = 0
        self.stop       = False

    def step(self, kappa):
        if kappa > self.best_kappa + self.min_delta:
            self.best_kappa = kappa
            self.counter    = 0
        else:
            self.counter += 1
            print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.stop = True

early_stopping = EarlyStopping(patience=5)

# Чекпоинты
def save_checkpoint(epoch, best_kappa, results, scheduler, scaler):
    local = '/content/ckpt_v4_last.pt'
    unfreeze_done = [k for k in globals().get('UNFREEZE_PLAN', {}).keys() if k < epoch]
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(),
                'best_kappa': best_kappa,
                'unfreeze_done': unfreeze_done}, local)
    shutil.copy(local, f'{CKPT_DIR}/ckpt_last.pt')
    with open(f'{RESULTS_DIR}/kappa_epoch_{epoch}.json','w') as f:
        json.dump(results, f, indent=2)
    print(f'  Сохранено: эпоха {epoch} | κ = {best_kappa:.4f}')

def load_checkpoint(scheduler, scaler, unfreeze_plan):
    path = f'{CKPT_DIR}/ckpt_last.pt'
    if not os.path.exists(path):
        print("Чекпоинт не найден — начинаем с нуля")
        return 0, 0.0, []

    try:
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
    except Exception as e:
        print(f"Внимание: Файл чекпоинта поврежден ({e}). Начинаем с нуля.")
        return 0, 0.0, []

    # Сначала восстановить размороженные слои в модели и добавить их в optimizer
    done = ckpt.get('unfreeze_done', [])
    for ep in sorted(done):
        if ep in unfreeze_plan:
            model.unfreeze_last_n(unfreeze_plan[ep], optimizer)

    # А теперь загрузить состояние оптимизатора, когда количество групп совпадает
    try:
        optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        if 'scaler' in ckpt:
            scaler.load_state_dict(ckpt['scaler'])
    except ValueError as e:
        print(f"Внимание: Не удалось загрузить состояние оптимизатора/scheduler/scaler ({e}). Будет использован чистый оптимизатор.")

    print(f"Загружен: эпоха {ckpt['epoch']} | κ = {ckpt['best_kappa']:.4f}")
    print(f"Разморожено при загрузке: {done}")
    return ckpt['epoch'], ckpt['best_kappa'], done

print('Оптимизатор, функция потерь и утилиты готовы.')

Оптимизатор, функция потерь и утилиты готовы.


## Шаг 7 — Оценка с TTA

In [ ]:
def find_best_threshold(probs, labels):
    best_k, best_t = -1.0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (np.array(probs) >= t).astype(int)
        if len(np.unique(preds)) < 2: continue
        k = cohen_kappa_score(labels, preds)
        if k > best_k: best_k, best_t = k, t
    return best_t, best_k


def evaluate(model, loader, use_tta=False):
    """Оценка с опциональным TTA (Test-Time Augmentation)."""
    model.eval()
    probs_by_bone  = defaultdict(list)
    labels_by_bone = defaultdict(list)

    if use_tta:
        # Создаём 4 версии val_dataset с разными аугментациями
        tta_loaders = [
            DataLoader(MURADataset(VAL_CSV, tta_idx=i),
                       batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS)
            for i in range(len(tta_transforms))
        ]
        # Собираем предсказания по всем TTA вариантам
        all_probs_tta = defaultdict(lambda: defaultdict(list))
        for tta_idx, tta_loader in enumerate(tta_loaders):
            with torch.no_grad():
                for images, labels, bones, study_ids in tta_loader:
                    images = images.to(DEVICE)
                    for bone in set(bones):
                        mask   = [i for i,b in enumerate(bones) if b == bone]
                        if not mask: continue
                        mask_t = torch.tensor(mask, device=DEVICE)
                        logits = model(images[mask_t], bone)
                        probs  = torch.softmax(logits,dim=1)[:,1].cpu().numpy()
                        all_probs_tta[bone][tta_idx].extend(probs.tolist())
                        if tta_idx == 0:
                            labels_by_bone[bone].extend(
                                [labels[i].item() for i in mask])
        # Усредняем по TTA
        for bone in BONE_CATEGORIES:
            if bone not in all_probs_tta: continue
            n = len(all_probs_tta[bone][0])
            avg = np.zeros(n)
            for tta_idx in range(len(tta_transforms)):
                avg += np.array(all_probs_tta[bone][tta_idx])
            probs_by_bone[bone] = (avg / len(tta_transforms)).tolist()
    else:
        with torch.no_grad():
            for images, labels, bones, _ in loader:
                images = images.to(DEVICE)
                for bone in set(bones):
                    mask   = [i for i,b in enumerate(bones) if b == bone]
                    if not mask: continue
                    mask_t = torch.tensor(mask, device=DEVICE)
                    logits = model(images[mask_t], bone)
                    probs  = torch.softmax(logits,dim=1)[:,1].cpu().numpy()
                    probs_by_bone[bone].extend(probs.tolist())
                    labels_by_bone[bone].extend(
                        [labels[i].item() for i in mask])

    results = {}
    all_optimized_preds = [] # Сюда будем складывать готовые предсказания (0 или 1)
    all_l = []               # Сюда истинные метки

    for bone in BONE_CATEGORIES:
        probs  = np.array(probs_by_bone.get(bone, []))
        labels = np.array(labels_by_bone.get(bone, []))

        if len(probs)==0 or len(np.unique(labels))<2:
            results[bone] = {'kappa':None,'auc':None,'f1':None,
                             'accuracy':None,'threshold':0.5,
                             'n_total':len(probs),'n_positive':0}
            continue

        best_t, best_k = find_best_threshold(probs, labels)
        preds  = (probs >= best_t).astype(int) # Предсказания по ИДЕАЛЬНОМУ порогу

        try: auc = float(roc_auc_score(labels, probs))
        except: auc = None

        results[bone] = {
            'kappa':    round(float(best_k), 4),
            'auc':      round(auc, 4) if auc else None,
            'f1':       round(float(f1_score(labels,preds,zero_division=0)),4),
            'accuracy': round(float(accuracy_score(labels,preds)),4),
            'threshold':round(float(best_t),2),
            'n_total':  int(len(labels)),
            'n_positive':int(labels.sum()),
        }

        # Сохраняем УЖЕ ПРИМЕНЕННЫЕ предсказания и метки для OVERALL
        all_optimized_preds.extend(preds.tolist())
        all_l.extend(labels.tolist())

    # Считаем OVERALL на основе индивидуальных порогов
    if all_l:
        overall_kappa = cohen_kappa_score(all_l, all_optimized_preds)
        overall_acc = accuracy_score(all_l, all_optimized_preds)
        overall_f1 = f1_score(all_l, all_optimized_preds, zero_division=0)

        # AUC считаем по сырым вероятностям, так как для него порог не нужен
        all_p_raw = []
        for bone in BONE_CATEGORIES:
            all_p_raw.extend(probs_by_bone.get(bone, []))
        try: auc_all = float(roc_auc_score(all_l, all_p_raw))
        except: auc_all = None

        results['OVERALL'] = {
            'kappa':    round(float(overall_kappa),4),
            'auc':      round(auc_all,4) if auc_all else None,
            'f1':       round(float(overall_f1),4),
            'accuracy': round(float(overall_acc),4),
            'threshold': -1.0,  # Обозначаем, что использовались смешанные пороги
            'n_total':  len(all_l),
            'n_positive':int(sum(all_l)),
        }
    return results


def print_table(results, title='', tta=False):
    tta_str = ' [+TTA]' if tta else ''
    print(f'\n{"="*72}')
    if title: print(f'  {title}{tta_str}')
    print(f'  {"Категория":<16} {"κ":>10} {"AUC":>7} {"F1":>7} {"Acc":>7} {"Порог":>7} {"N":>6}')
    print(f'  {"-"*70}')
    for bone in BONE_CATEGORIES:
        r    = results.get(bone, {})
        k    = f"{r['kappa']:.4f}" if r.get('kappa') is not None else '  N/A '
        auc  = f"{r['auc']:.4f}"   if r.get('auc')   is not None else ' N/A '
        f1   = f"{r['f1']:.4f}"    if r.get('f1')    is not None else ' N/A '
        acc  = f"{r['accuracy']:.4f}" if r.get('accuracy') is not None else ' N/A '
        t    = f"{r.get('threshold',0.5):.2f}"
        n    = r.get('n_total',0)
        flag = ' ←' if bone in WEAK_BONES else ''
        print(f'  {bone:<16} {k:>10} {auc:>7} {f1:>7} {acc:>7} {t:>7} {n:>6}{flag}')
    print(f'  {"-"*70}')
    r = results.get('OVERALL', {})
    if r:
        print(f'  {"OVERALL":<16} {r.get("kappa",0):>10.4f} '
              f'{r.get("auc",0):>7.4f} {r.get("f1",0):>7.4f} '
              f'{r.get("accuracy",0):>7.4f} '
              f'{r.get("threshold",0.5):>7.2f} {r.get("n_total",0):>6}')
    print(f'{"="*72}\n')

print('Функции оценки с TTA готовы')


Функции оценки с TTA готовы


In [ ]:
import os
import shutil

# Очищаем папку с чекпоинтами от старых экспериментов
if os.path.exists(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists('/content/ckpt_v4_last.pt'):
    os.remove('/content/ckpt_v4_last.pt')

print("Папка с чекпоинтами очищена!")

Папка с чекпоинтами очищена!


## Шаг 8 — Тренировочный цикл (ProgressiveLoss)

In [ ]:
experiment_history = []

# ─── ПЛАН РАЗМОРАЖИВАНИЯ ───────────────────────
UNFREEZE_PLAN = {
    4:  4,
    8:  8,
    11: 12,
    15: 16,
}

from transformers import get_cosine_schedule_with_warmup

total_steps  = EPOCHS * len(train_loader) // ACCUM_STEPS
warmup_steps = int(total_steps * 0.08)  # 8% на разогрев

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda')

# Загружаем последний сохраненный чекпоинт
START_EPOCH, best_kappa, unfreeze_done = load_checkpoint(scheduler, scaler, UNFREEZE_PLAN)

# Загрузка best_model.pt с защитой от несовпадения размерностей
if os.path.exists(f'{CKPT_DIR}/best_model.pt'):
    state_dict = torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE)
    model_dict = model.state_dict()

    # Оставляем только те веса, размер которых совпадает
    pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.size() == model_dict[k].size()}

    if len(pretrained_dict) < len(state_dict):
        print("Внимание: Архитектура изменилась (например, добавлен Patch Pooling)! Загружены только совпадающие веса.")
        print("Принудительный старт с 1-й эпохи для обучения новой головы.")
        START_EPOCH = 0
    else:
        print("Загружены веса из best_model.pt.")

    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

early_stopping.best_kappa = best_kappa

print(f'Обучение SimpleMURA (DINOv2) [Эпохи {START_EPOCH+1}..{EPOCHS}]')
print(f'Batch: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}\n')

for epoch in range(START_EPOCH, EPOCHS):
    criterion.set_epoch(epoch)
    phase = 'BCE' if epoch < 5 else f'Mix({(epoch-5)/4:.1f})' if epoch < 9 else 'Focal'
    print(f"\nЭпоха {epoch+1}: Loss = {phase}")

    if epoch in UNFREEZE_PLAN and epoch not in unfreeze_done:
        n = UNFREEZE_PLAN[epoch]
        print(f"  [Эпоха {epoch+1}] Размораживаем последние {n} блоков...")
        model.unfreeze_last_n(n, optimizer)
        remaining    = EPOCHS - epoch
        remain_steps = remaining * len(train_loader) // ACCUM_STEPS
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(remain_steps * 0.05),
            num_training_steps=remain_steps
        )
        early_stopping.counter    = 0
        early_stopping.best_kappa = best_kappa
        unfreeze_done.append(epoch)

    t0 = time.time()
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (images, labels, bones, _) in enumerate(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        unique_bones = list(set(bones))

        loss = torch.tensor(0.0, device=DEVICE)
        n_bone = 0

        for bone in unique_bones:
            mask = [i for i,b in enumerate(bones) if b==bone]
            if not mask: continue
            mask_t = torch.tensor(mask, device=DEVICE)
            imgs_b = images[mask_t]
            labs_b = labels[mask_t]
            bone_w = BONE_LOSS_WEIGHTS.get(bone, 1.0)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs_b, bone)

            loss_part = bone_w * criterion(logits.float(), labs_b)
            loss += loss_part
            n_bone += 1

        if n_bone == 0: continue

        loss = loss / n_bone / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 100 == 0:
            vram = torch.cuda.memory_allocated()/1e9
            lr   = optimizer.param_groups[-1]['lr']
            print(f'  [{epoch+1:02d}/{EPOCHS}] шаг {step:04d} loss={running_loss/(step+1):.4f} lr={lr:.2e} VRAM={vram:.1f}GB')

    avg_loss  = running_loss / len(train_loader)
    epoch_min = (time.time() - t0) / 60

    results = evaluate(model, val_loader)
    print_table(results, title=f'Эпоха {epoch+1} | loss={avg_loss:.4f}')

    kappa = results.get('OVERALL', {}).get('kappa') or 0.0
    row = {'epoch': epoch+1, 'train_loss': round(avg_loss, 4), 'time_min': round(epoch_min, 1)}
    for bone in BONE_CATEGORIES:
        row[f'kappa_{bone}'] = results.get(bone, {}).get('kappa')
    row['kappa_OVERALL'] = kappa
    experiment_history.append(row)

    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pt')
        print(f'  ⭐ Новый лучший: κ = {best_kappa:.4f}')

    if (epoch+1) % 2 == 0:
        save_checkpoint(epoch+1, best_kappa, results, scheduler, scaler)

    print(f'  {epoch_min:.1f} мин | κ={kappa:.4f}\n')
    early_stopping.step(kappa)
    if early_stopping.stop:
        print(f'Early stopping эпоха {epoch+1}')
        break

print(f'Обучение завершено. Лучшая каппа: {best_kappa:.4f}')

Чекпоинт не найден — начинаем с нуля
Обучение SimpleMURA (DINOv2) [Эпохи 1..18]
Batch: 16 x 4 = 64


Эпоха 1: Loss = BCE
  [01/18] шаг 0000 loss=0.9463 lr=0.00e+00 VRAM=1.3GB
  [01/18] шаг 0100 loss=1.0480 lr=1.01e-05 VRAM=1.3GB
  [01/18] шаг 0200 loss=1.0305 lr=2.01e-05 VRAM=1.3GB
  [01/18] шаг 0300 loss=1.0160 lr=3.02e-05 VRAM=1.3GB
  [01/18] шаг 0400 loss=1.0094 lr=4.03e-05 VRAM=1.3GB
  [01/18] шаг 0500 loss=1.0021 lr=5.03e-05 VRAM=1.3GB
  [01/18] шаг 0600 loss=0.9968 lr=6.04e-05 VRAM=1.3GB
  [01/18] шаг 0700 loss=0.9941 lr=7.05e-05 VRAM=1.3GB
  [01/18] шаг 0800 loss=0.9861 lr=8.05e-05 VRAM=1.3GB
  [01/18] шаг 0900 loss=0.9798 lr=9.06e-05 VRAM=1.3GB
  [01/18] шаг 1000 loss=0.9735 lr=1.01e-04 VRAM=1.3GB
  [01/18] шаг 1100 loss=0.9710 lr=1.11e-04 VRAM=1.3GB
  [01/18] шаг 1200 loss=0.9647 lr=1.21e-04 VRAM=1.3GB
  [01/18] шаг 1300 loss=0.9603 lr=1.31e-04 VRAM=1.3GB
  [01/18] шаг 1400 loss=0.9580 lr=1.41e-04 VRAM=1.3GB
  [01/18] шаг 1500 loss=0.9550 lr=1.51e-04 VRAM=1.3GB
  [01/18] шаг 1

In [ ]:
# Шаг 1 — загрузить лучшую модель
model.load_state_dict(
    torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE)
)
model.eval()
print(f'✅ Загружена лучшая модель (κ train/val = {best_kappa:.4f})')

# Шаг 2 — оценка на вашем val (10% от train пациентов)
print('\n📊 Оценка на VAL (10% от train):')
results_val = evaluate(model, val_loader, use_tta=False)
print_table(results_val, title='Val (10% пациентов из train)')
k_val = results_val.get('OVERALL', {}).get('kappa', 0)

# Шаг 3 — оценка на официальном Stanford test
print('\n📊 Оценка на официальном Stanford TEST:')
results_test = evaluate(model, test_loader, use_tta=False)
print_table(results_test, title='Stanford Test Set (официальный)')
k_test = results_test.get('OVERALL', {}).get('kappa', 0)


# Шаг 5 — итоговое сравнение
print('  ИТОГ:')
print('='*55)
print(f'  Val (10% пациентов):      κ = {k_val:.4f}')
print(f'  Stanford Test (без TTA):  κ = {k_test:.4f}')
print(f'  Предыдущий лучший (448):  κ = 0.7112')
print(f'  Разница:                  {k_test - 0.7112:+.4f}')


✅ Загружена лучшая модель (κ train/val = 0.7439)

📊 Оценка на VAL (10% от train):

  Val (10% пациентов из train)
  Категория                 κ     AUC      F1     Acc   Порог      N
  ----------------------------------------------------------------------
  XR_WRIST             0.8009  0.9419  0.8850  0.9025    0.56    954
  XR_ELBOW             0.7745  0.9360  0.8564  0.8954    0.35    497
  XR_SHOULDER          0.6667  0.8979  0.8158  0.8348    0.65    817
  XR_FINGER            0.7030  0.9063  0.8131  0.8612    0.64    533 ←
  XR_FOREARM           0.8368  0.9376  0.8889  0.9290    0.11    155 ←
  XR_HUMERUS           0.8999  0.9791  0.9483  0.9500    0.57    120
  XR_HAND              0.6624  0.8737  0.7454  0.8748    0.47    551 ←
  ----------------------------------------------------------------------
  OVERALL              0.7439  0.9212  0.8421  0.8787   -1.00   3627


📊 Оценка на официальном Stanford TEST:

  Stanford Test Set (официальный)
  Категория                 κ     AUC

## Шаг 9 — Финальная оценка с TTA

In [ ]:
# Загрузить лучшую модель
model.load_state_dict(torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE))

# Оценка без TTA
results_no_tta = evaluate(model, val_loader, use_tta=False)
print_table(results_no_tta, title='Без TTA')

print('\nЗапускаем TTA (4 аугментации × весь val)...')
results_tta = evaluate(model, val_loader, use_tta=True)
print_table(results_tta, title='С TTA (4 аугментации)', tta=True)

k_no_tta = results_no_tta.get('OVERALL',{}).get('kappa',0)
k_tta    = results_tta.get('OVERALL',{}).get('kappa',0)
print(f'TTA прирост: {k_tta - k_no_tta:+.4f}')


  Без TTA
  Категория                 κ     AUC      F1     Acc   Порог      N
  ----------------------------------------------------------------------
  XR_WRIST             0.8009  0.9419  0.8850  0.9025    0.56    954
  XR_ELBOW             0.7745  0.9360  0.8564  0.8954    0.35    497
  XR_SHOULDER          0.6667  0.8979  0.8158  0.8348    0.65    817
  XR_FINGER            0.7030  0.9063  0.8131  0.8612    0.64    533 ←
  XR_FOREARM           0.8368  0.9376  0.8889  0.9290    0.11    155 ←
  XR_HUMERUS           0.8999  0.9791  0.9483  0.9500    0.57    120
  XR_HAND              0.6624  0.8737  0.7454  0.8748    0.47    551 ←
  ----------------------------------------------------------------------
  OVERALL              0.7439  0.9212  0.8421  0.8787   -1.00   3627


Запускаем TTA (4 аугментации × весь val)...

  С TTA (4 аугментации) [+TTA]
  Категория                 κ     AUC      F1     Acc   Порог      N
  -------------------------------------------------------------------

In [ ]:
import os

# Проверка на случай, если TTA был пропущен или прерван
k_tta_val = globals().get('k_tta', 0)
k_no_tta_val = globals().get('k_no_tta', results_no_tta.get('OVERALL',{}).get('kappa',0))
res_tta_val = globals().get('results_tta', results_no_tta)

# Сохранить финальную модель с порогами
final = res_tta_val if k_tta_val > k_no_tta_val else results_no_tta
best_method = 'TTA' if k_tta_val > k_no_tta_val else 'No TTA'

thresholds = {
    bone: final.get(bone, {}).get('threshold', 0.5)
    for bone in BONE_CATEGORIES
}

os.makedirs(f'{CKPT_DIR}/inference', exist_ok=True)

torch.save({
    'model_state':     model.state_dict(),
    'thresholds':      thresholds,
    'bone_categories': BONE_CATEGORIES,
    'best_kappa':      max(k_no_tta_val, k_tta_val),
    'method':          best_method,
}, f'{CKPT_DIR}/inference/mura_dinov2_complete.pt')

print(f"\n✅ Модель сохранена для инференса")
print(f"   Каппа:   {max(k_no_tta_val, k_tta_val):.4f}")
print(f"   Метод:   {best_method}")
print(f"   Пороги:  {thresholds}")

In [ ]:
import shutil
import os

# Путь к сохраненной модели с TTA-порогами
source_inference = f'{CKPT_DIR}/inference/mura_dinov2_complete.pt'
# Путь для бэкапа в корне Диска
backup_inference = '/content/drive/MyDrive/mura_dinov2_complete_TTA.pt'

if os.path.exists(source_inference):
    shutil.copy(source_inference, backup_inference)
    size_mb = os.path.getsize(backup_inference) / (1024 * 1024)
    print(f"\n✅ Успех! Чекпоинт с порогами TTA скопирован.")
    print(f"📁 Найти его можно тут: {backup_inference}")
    print(f"📊 Размер файла: {size_mb:.1f} MB")
else:
    print(f"\n⚠️ Файл {source_inference} не найден.")

## Шаг 10 — Подбор оптимальных порогов (Cross-Validation)
Убираем оптимистичный bias при поиске порога, разделяя валидационный сет через `StratifiedKFold`.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import torch
import os

def find_optimal_thresholds(model, val_loader, n_splits=5):
    model.eval()
    probs_by_bone  = defaultdict(list)
    labels_by_bone = defaultdict(list)

    # Собрать все предсказания - ИСПРАВЛЕНО: распаковка 4 значений
    with torch.no_grad():
        for images, labels, bones, study_ids in val_loader:
            images = images.to(DEVICE)
            for bone in set(bones):
                mask   = [i for i,b in enumerate(bones) if b==bone]
                if not mask: continue
                mask_t = torch.tensor(mask, device=DEVICE)
                logits = model(images[mask_t], bone)
                probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

                probs_by_bone[bone].extend(probs.tolist())
                labels_by_bone[bone].extend([labels[i].item() for i in mask])

    thresholds = {}
    results    = {}

    print(f"\n{'='*60}")
    print(f"  Поиск оптимальных порогов (CV на val set)")
    print(f"{'='*60}")
    print(f"  {'Категория':<16} {'Порог':>7} {'κ (CV)':>8} {'κ (full)':>9} {'N':>6}")
    print(f"  {'-'*56}")

    for bone in BONE_CATEGORIES:
        probs  = np.array(probs_by_bone.get(bone, []))
        labels = np.array(labels_by_bone.get(bone, []))

        if len(probs) == 0 or len(np.unique(labels)) < 2:
            thresholds[bone] = 0.5
            continue

        # ── CV поиск порога ───────────────────────────────
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        threshold_candidates = np.arange(0.05, 0.95, 0.01)
        cv_kappas = np.zeros(len(threshold_candidates))

        for train_idx, val_idx in skf.split(probs, labels):
            p_train = probs[train_idx]
            l_train = labels[train_idx]

            for i, t in enumerate(threshold_candidates):
                preds = (p_train >= t).astype(int)
                if len(np.unique(preds)) < 2: continue
                cv_kappas[i] += cohen_kappa_score(l_train, preds)

        cv_kappas /= n_splits
        best_idx   = np.argmax(cv_kappas)
        best_t     = threshold_candidates[best_idx]
        best_k_cv  = cv_kappas[best_idx]

        # Оценить на полном val
        preds_full = (probs >= best_t).astype(int)
        k_full     = cohen_kappa_score(labels, preds_full)

        thresholds[bone] = round(float(best_t), 2)
        results[bone]    = {
            'threshold':  round(float(best_t), 2),
            'kappa_cv':   round(float(best_k_cv), 4),
            'kappa_full': round(float(k_full), 4),
            'n':          len(labels),
        }

        print(f"  {bone:<16} {best_t:>7.2f} {best_k_cv:>8.4f} {k_full:>9.4f} {len(labels):>6}")

    # OVERALL порог
    all_probs, all_labels = [], []
    for bone in BONE_CATEGORIES:
        all_probs.extend(probs_by_bone.get(bone, []))
        all_labels.extend(labels_by_bone.get(bone, []))

    if all_labels:
        all_p = np.array(all_probs)
        all_l = np.array(all_labels)
        best_k, best_t = -1, 0.5
        for t in np.arange(0.05, 0.95, 0.01):
            preds = (all_p >= t).astype(int)
            if len(np.unique(preds)) < 2: continue
            k = cohen_kappa_score(all_l, preds)
            if k > best_k:
                best_k, best_t = k, t
        thresholds['OVERALL'] = round(float(best_t), 2)
        print(f"  {'-'*56}")
        print(f"  {'OVERALL':<16} {best_t:>7.2f} {'':>8} {best_k:>9.4f} {len(all_labels):>6}")
    print(f"{'='*60}")
    return thresholds, results

def predict_with_thresholds(model, loader, thresholds):
    model.eval()
    all_preds, all_labels, all_bones  = [], [], []
    with torch.no_grad():
        for images, labels, bones, study_ids in loader:
            images = images.to(DEVICE)
            for bone in set(bones):
                mask   = [i for i,b in enumerate(bones) if b==bone]
                if not mask: continue
                mask_t = torch.tensor(mask, device=DEVICE)
                logits = model(images[mask_t], bone)
                probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
                t      = thresholds.get(bone, thresholds.get('OVERALL', 0.5))
                preds  = (probs >= t).astype(int)
                all_preds.extend(preds.tolist())
                all_labels.extend([labels[i].item() for i in mask])
                all_bones.extend([bone] * len(mask))

    print(f"\n{'='*55}")
    print(f"  Результаты с оптимальными порогами")
    print(f"{'='*55}")
    print(f"  {'Категория':<16} {'Порог':>7} {'κ':>8} {'N':>6}")
    print(f"  {'-'*50}")

    all_p_arr = np.array(all_preds)
    all_l_arr = np.array(all_labels)
    all_b_arr = np.array(all_bones)

    for bone in BONE_CATEGORIES:
        mask = all_b_arr == bone
        if mask.sum() == 0: continue
        p, l, t = all_p_arr[mask], all_l_arr[mask], thresholds.get(bone, 0.5)
        if len(np.unique(l)) < 2:
            print(f"  {bone:<16} {t:>7.2f} {'N/A':>8}")
            continue
        print(f"  {bone:<16} {t:>7.2f} {cohen_kappa_score(l, p):>8.4f} {mask.sum():>6}")

    print(f"  {'-'*50}")
    if len(np.unique(all_l_arr)) >= 2:
        k_overall = cohen_kappa_score(all_l_arr, all_p_arr)
        t_overall = thresholds.get('OVERALL', 0.5)
        print(f"  {'OVERALL':<16} {t_overall:>7.2f} {k_overall:>8.4f} {len(all_l_arr):>6}")
    print(f"{'='*55}")
    return all_preds, all_labels

# ─── ИСПОЛЬЗОВАНИЕ ─────────────────────────────────────────────
print("Загрузка лучшей модели...")
model.load_state_dict(torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE))

cv_thresholds, thresh_results = find_optimal_thresholds(model, val_loader, n_splits=5)
preds, labels = predict_with_thresholds(model, val_loader, cv_thresholds)

os.makedirs(f'{CKPT_DIR}/inference', exist_ok=True)
torch.save({
    'model_state':     model.state_dict(),
    'thresholds':      cv_thresholds,
    'bone_categories': BONE_CATEGORIES,
    'best_kappa':      globals().get('best_kappa', 0.0),
}, f'{CKPT_DIR}/inference/final_with_cv_thresholds.pt')

print(f"\n✅ Модель и оптимальные пороги сохранены в final_with_cv_thresholds.pt")

Загрузка лучшей модели...

  Поиск оптимальных порогов (CV на val set)
  Категория          Порог   κ (CV)  κ (full)      N
  --------------------------------------------------------
  XR_WRIST            0.56   0.8009    0.8009    954


## Шаг 11 — Дообучение (Fine-Tuning) с исправленным LR
Исправление зависания каппы (увеличен LR для backbone, смягчена gamma для Focal Loss).

In [ ]:
model.load_state_dict(torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE))

# Разморозить 12 блоков
layers = list(model.dino.encoder.layer)
for layer in layers[-12:]:
    for p in layer.parameters():
        p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
print(f"Обучаемых: {trainable:.1f}M")

Обучаемых: 305.5M


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# Разделить backbone и head
backbone_params = [p for n,p in model.named_parameters()
                   if p.requires_grad and 'dino' in n]
head_params     = [p for n,p in model.named_parameters()
                   if p.requires_grad and 'dino' not in n]

print(f"backbone params: {sum(p.numel() for p in backbone_params)/1e6:.1f}M")
print(f"head params:     {sum(p.numel() for p in head_params)/1e6:.1f}M")

# ИСПРАВЛЕНО: LR в 5-10x больше чем было
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 8e-6, 'name': 'backbone'},
    {'params': head_params,     'lr': 5e-5, 'name': 'head'},
], weight_decay=0.01)

# Scheduler на 10 новых эпох
EXTRA_EPOCHS = 10
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EXTRA_EPOCHS * len(train_loader) // ACCUM_STEPS,
    eta_min=1e-8
)


print(f"Epochs:      {EXTRA_EPOCHS}")

In [ ]:
# gamma=2.0 слишком агрессивный — переобучение
# Вернуть к 1.5 и заморозить на Focal фазе
criterion.gamma    = 1.5
criterion.epoch    = 9   # сразу в Focal режиме
print(f"gamma: {criterion.gamma} (было 2.0)")
print(f"phase: Focal")

In [ ]:
import time
from collections import defaultdict
import numpy as np
import torch
# Пересоздаем EarlyStopping с нужным best_kappa
early_stopping = EarlyStopping(patience=5)
early_stopping.best_kappa = best_kappa

START_EPOCH = 0
EXTRA_EPOCHS = 10
scaler = torch.amp.GradScaler('cuda')

print(f"Продолжаем с κ={best_kappa:.4f}")
print(f"backbone LR=8e-6, head LR=5e-5, gamma=1.5\n")

for epoch in range(EXTRA_EPOCHS):
    criterion.set_epoch(9 + epoch)  # всегда Focal
    t0 = time.time()
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (images, labels, bones) in enumerate(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        unique_bones = list(set(bones))

        loss   = torch.tensor(0.0, device=DEVICE, dtype=torch.float32)
        n_bone = 0

        for bone in unique_bones:
            mask   = [i for i,b in enumerate(bones) if b == bone]
            if not mask: continue
            mask_t = torch.tensor(mask, device=DEVICE)
            imgs_b = images[mask_t]
            labs_b = labels[mask_t]
            bone_w = BONE_LOSS_WEIGHTS.get(bone, 1.0)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs_b, bone)

            # criterion СНАРУЖИ autocast
            loss_part = bone_w * criterion(logits.float(), labs_b)
            loss   += loss_part
            n_bone += 1

        if n_bone == 0: continue
        loss = loss / n_bone / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 100 == 0:
            vram = torch.cuda.memory_allocated()/1e9
            lr_b = optimizer.param_groups[0]['lr']
            lr_h = optimizer.param_groups[1]['lr']
            avg  = running_loss/(step+1)
            print(f"  [{epoch+1:02d}/{EXTRA_EPOCHS}] шаг {step:04d} "
                  f"loss={avg:.4f} lr_b={lr_b:.2e} lr_h={lr_h:.2e} "
                  f"VRAM={vram:.1f}GB")

    avg_loss  = running_loss / len(train_loader)
    epoch_min = (time.time()-t0)/60

    results = evaluate(model, val_loader, use_tta=False)
    print_table(results, title=f'Эпоха +{epoch+1} | loss={avg_loss:.4f}')

    kappa = results.get('OVERALL', {}).get('kappa') or 0.0

    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pt')
        print(f"  ⭐ Новый лучший: κ = {best_kappa:.4f}")

    print(f"  {epoch_min:.1f} мин | κ={kappa:.4f}\n")

    early_stopping.step(kappa)
    if early_stopping.stop:
        print(f"  Early stopping на эпохе +{epoch+1}")
        break

print(f"\nЛучшая каппа: {best_kappa:.4f}")

## Шаг 12 — Резервное копирование чекпоинта
Эта ячейка скопирует лучший чекпоинт в корень вашего Google Диска, чтобы он точно никуда не пропал.

In [ ]:
import shutil
import os

# Путь, куда сохраняется лучшая модель во время обучения
source_path = f'{CKPT_DIR}/best_model.pt'
# Куда скопировать (в корень вашего Google Диска)
backup_path = '/content/drive/MyDrive/MURA_best_model_backup.pt'

print("Проверяем наличие сохраненного чекпоинта...")
if os.path.exists(source_path):
    shutil.copy(source_path, backup_path)
    size_mb = os.path.getsize(backup_path) / (1024 * 1024)
    print(f"\n✅ Успех! Лучший чекпоинт надежно скопирован.")
    print(f"📁 Найти его можно тут: {backup_path}")
    print(f"📊 Размер файла: {size_mb:.1f} MB")
    print("Теперь вы можете безопасно закрывать сессию Colab.")
else:
    print(f"\n⚠️ Файл {source_path} не найден.")
    print("Возможно, метрика не улучшилась за время дообучения, или Диск отвалился.")

Проверяем наличие сохраненного чекпоинта...

✅ Успех! Лучший чекпоинт надежно скопирован.
📁 Найти его можно тут: /content/drive/MyDrive/MURA_best_model_backup.pt
📊 Размер файла: 1165.4 MB
Теперь вы можете безопасно закрывать сессию Colab.


 Full Fine-Tuning Strategy


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleBCE(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        # Standard CrossEntropy with weighted support for MURA imbalance
        self.pos_weight = pos_weight

    def forward(self, logits, labels):
        weights = torch.tensor([1.0, self.pos_weight], dtype=torch.float, device=logits.device)
        return F.cross_entropy(logits.float(), labels, weight=weights)

In [ ]:
import os

# Ensure model is initialized
if 'model' not in globals():
    print('Инициализация модели SimpleMURA...')
    model = SimpleMURA(BONE_CATEGORIES).to(DEVICE)

# 1. Fully unfreeze all parameters for Stage 2
for p in model.parameters():
    p.requires_grad = True

# 2. Group parameters for differential LR
param_groups = [
    {'params': [], 'lr': 5e-7, 'name': 'early_layers'},
    {'params': [], 'lr': 2e-6, 'name': 'mid_layers'},
    {'params': [], 'lr': 5e-6, 'name': 'late_layers'},
    {'params': [], 'lr': 5e-5, 'name': 'head_and_cls'}
]

for name, param in model.named_parameters():
    if not param.requires_grad: continue
    if 'head' in name or 'classifiers' in name:
        param_groups[3]['params'].append(param)
    elif 'dino.encoder.layer' in name:
        layer_idx = int(name.split('.')[3])
        if layer_idx < 12:
            param_groups[0]['params'].append(param)
        elif layer_idx < 20:
            param_groups[1]['params'].append(param)
        else:
            param_groups[2]['params'].append(param)
    else:
        param_groups[0]['params'].append(param)

optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)

# 3. Resume logic
START_EPOCH_S2 = 0
best_kappa = globals().get('best_kappa', 0.0)
CKPT_STAGE2 = f'{CKPT_DIR}/ckpt_last_stage2.pt'

if os.path.exists(CKPT_STAGE2):
    print(f'🔄 Восстановление Stage 2 из чекпоинта...')
    ckpt = torch.load(CKPT_STAGE2, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    START_EPOCH_S2 = ckpt['epoch']
    best_kappa = ckpt['best_kappa']
    print(f'✅ Возобновляем с эпохи {START_EPOCH_S2+1} (best kappa {best_kappa:.4f})')
else:
    try:
        model.load_state_dict(torch.load(f'{CKPT_DIR}/best_model.pt', map_location=DEVICE))
        print(f'✅ Загружена лучшая модель Этапа 1 для начала Этапа 2')
    except:
        print('⚠️ Веса Этапа 1 не найдены, продолжаем с текущим состоянием')

print(f'Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

✅ Загружена лучшая модель Этапа 1 для начала Этапа 2
Total trainable parameters: 305.5M


In [ ]:
EXTRA_EPOCHS = 6

# Ensure SimpleBCE is defined
if 'SimpleBCE' not in globals():
    class SimpleBCE(nn.Module):
        def __init__(self, pos_weight):
            super().__init__()
            self.pos_weight = pos_weight
        def forward(self, logits, labels):
            weights = torch.tensor([1.0, self.pos_weight], dtype=torch.float, device=logits.device)
            return F.cross_entropy(logits.float(), labels, weight=weights)

criterion = SimpleBCE(pos_weight=POS_WEIGHT.item())

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EXTRA_EPOCHS * len(train_loader) // ACCUM_STEPS,
    eta_min=1e-8
)
scaler = torch.amp.GradScaler('cuda')

if START_EPOCH_S2 > 0:
    print(f'Промотка планировщика до шага {START_EPOCH_S2 * len(train_loader) // ACCUM_STEPS}')
    for _ in range(START_EPOCH_S2 * len(train_loader) // ACCUM_STEPS):
        scheduler.step()

print(f'🚀 Старт тренировки Stage 2: Эпохи {START_EPOCH_S2+1} - {EXTRA_EPOCHS}')

for epoch in range(START_EPOCH_S2, EXTRA_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for step, (images, labels, bones, _) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        unique_bones = list(set(bones))

        loss = torch.tensor(0.0, device=DEVICE)
        n_bone = 0

        for bone in unique_bones:
            mask = [i for i, b in enumerate(bones) if b == bone]
            if not mask: continue
            mask_t = torch.tensor(mask, device=DEVICE)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images[mask_t], bone)
                bone_w = BONE_LOSS_WEIGHTS.get(bone, 1.0)
                loss_part = bone_w * criterion(logits, labels[mask_t])

            loss += loss_part
            n_bone += 1

        loss = loss / n_bone / ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            # ИСПРАВЛЕНО: шаг планировщика ПОСЛЕ шага оптимизатора
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUM_STEPS

    # Validation
    results = evaluate(model, val_loader)
    kappa = results.get('OVERALL', {}).get('kappa', 0.0)

    print_table(results, title=f'Stage 2 | Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f}')

    # Save checkpoint
    torch.save({
        'epoch': epoch + 1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_kappa': best_kappa
    }, CKPT_STAGE2)

    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pt')
        print(f'  ⭐ Новый рекорд Kappa: {best_kappa:.4f}')

    print(f'  Эпоха {epoch+1} за {(time.time()-t0)/60:.1f} мин')

🚀 Старт тренировки Stage 2: Эпохи 1 - 6

  Stage 2 | Epoch 1 | Loss: 0.5593
  Категория                 κ     AUC      F1     Acc   Порог      N
  ----------------------------------------------------------------------
  XR_WRIST             0.7786  0.9427  0.8698  0.8920    0.49    954
  XR_ELBOW             0.7518  0.9387  0.8499  0.8813    0.17    497
  XR_SHOULDER          0.6395  0.8860  0.8118  0.8201    0.36    817
  XR_FINGER            0.6900  0.8931  0.7989  0.8574    0.74    533 ←
  XR_FOREARM           0.8293  0.9447  0.8791  0.9290    0.89    155 ←
  XR_HUMERUS           0.9333  0.9722  0.9655  0.9667    0.36    120
  XR_HAND              0.7082  0.8815  0.7724  0.8984    0.54    551 ←
  ----------------------------------------------------------------------
  OVERALL              0.7349  0.9184  0.8369  0.8743   -1.00   3627

  ⭐ Новый рекорд Kappa: 0.7349
  Эпоха 1 за 21.8 мин

  Stage 2 | Epoch 2 | Loss: 0.5086
  Категория                 κ     AUC      F1     Acc   Порог

In [ ]:
from google.colab import drive
drive.mount('/content/drive')